# 03 — Feature Engineering

## Objective
Transform the raw Vehicle Insurance Fraud dataset into a reproducible, model-ready feature dataset. We retain useful information, create defensible derived features, remove identifier-like fields, and split data before any learned preprocessing.

**Important:** Encoders, imputers, and scalers are not fitted here. They will be fitted only on training data inside model pipelines to prevent leakage.

## 1. Imports and project paths
The path helper supports running this notebook from either the project root or the `notebooks/` directory.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find the project root by walking up from the notebook working directory.
CWD = Path.cwd()
PROJECT_ROOT = next(
    (candidate for candidate in [CWD, *CWD.parents] if (candidate / 'data' / 'raw' / 'fraud_oracle.csv').exists()),
    None
)
if PROJECT_ROOT is None:
    # Typical fallback when notebook is launched from notebooks/ before data is visible.
    PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'fraud_oracle.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Raw data path:', RAW_PATH)

Project root: c:\Users\rudra\OneDrive\Documents\insurance-claim-fraud-detection\insurance-claim-fraud-detection
Raw data path: c:\Users\rudra\OneDrive\Documents\insurance-claim-fraud-detection\insurance-claim-fraud-detection\data\raw\fraud_oracle.csv


## 2. Load raw data and verify target

In [2]:
df = pd.read_csv(RAW_PATH)
TARGET = 'FraudFound_P'
assert TARGET in df.columns, f'Missing target column: {TARGET}'
print('Shape:', df.shape)
print('Target values:', sorted(df[TARGET].dropna().unique()))
df.head()

Shape: (15420, 33)
Target values: [np.int64(0), np.int64(1)]


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


## 3. Separate target from predictors
`FraudFound_P` is our target (0 = legitimate, 1 = fraud). We keep it separate so it cannot accidentally become a predictor.

In [3]:
y = df[TARGET].astype(int).copy()
X_raw = df.drop(columns=[TARGET]).copy()
print('X shape:', X_raw.shape)
print('y shape:', y.shape)
print('Fraud prevalence:', round(y.mean() * 100, 2), '%')

X shape: (15420, 32)
y shape: (15420,)
Fraud prevalence: 5.99 %


## 4. Remove identifier-like fields
`PolicyNumber` is an identifier and `RepNumber` represents a claim representative. We exclude them from this initial modelling feature set to avoid arbitrary identifier patterns and representative-specific effects. They may be retained separately for operational traceability.

In [4]:
identifier_cols = ['PolicyNumber', 'RepNumber']
existing_identifier_cols = [c for c in identifier_cols if c in X_raw.columns]
X = X_raw.drop(columns=existing_identifier_cols).copy()
print('Dropped identifier-like fields:', existing_identifier_cols)
print('Remaining predictors:', X.shape[1])

Dropped identifier-like fields: ['PolicyNumber', 'RepNumber']
Remaining predictors: 30


## 5. Review feature types
Integer fields such as week numbers and driver ratings may be categorical or ordinal rather than continuous. This review helps inform preprocessing.

In [5]:
feature_type_summary = pd.DataFrame({
    'dtype': X.dtypes.astype(str),
    'n_unique': X.nunique(dropna=False),
    'example_values': [X[c].dropna().unique()[:5].tolist() for c in X.columns]
}).sort_values('dtype')
feature_type_summary

,dtype,n_unique,example_values
WeekOfMonth,int64,5,"[5, 3, 2, 4, 1]"
WeekOfMonthClaimed,int64,5,"[1, 4, 2, 3, 5]"
Age,int64,66,"[21, 34, 47, 65, 27]"
Deductible,int64,4,"[300, 400, 500, 700]"
DriverRating,int64,4,"[1, 4, 3, 2]"
Year,int64,3,"[1994, 1995, 1996]"
Month,str,12,"[Dec, Jan, Oct, Jun, Feb]"
DayOfWeek,str,7,"[Wednesday, Friday, Saturday, Monday, Tuesday]"
Make,str,19,"[Honda, Toyota, Ford, Mazda, Chevrolet]"
AccidentArea,str,2,"[Urban, Rural]"


## 6. Clean placeholders and engineer features
We replace the placeholder value `0` in claimed month/day fields with missing values. We create mismatch indicators only where the claimed field is valid. Exact elapsed time is not calculated from month/week fields because calendar boundaries and the meaning of `Year` require validation.

In [6]:
# Clean placeholder values before comparisons
for col in ['MonthClaimed', 'DayOfWeekClaimed']:
    if col in X.columns:
        X[col] = X[col].replace('0', np.nan)

# Age group from actual claimant age
X['AgeGroup'] = pd.cut(
    X['Age'], bins=[-1, 20, 25, 30, 35, 40, 50, 65, np.inf],
    labels=['<=20', '21-25', '26-30', '31-35', '36-40', '41-50', '51-65', '65+'],
    right=True
).astype('object')

# Mismatch indicators (missing claimed values remain missing)
X['AccidentClaimMonthMismatch'] = np.where(
    X['MonthClaimed'].notna(), (X['Month'] != X['MonthClaimed']).astype(int), np.nan
)
X['AccidentClaimDayMismatch'] = np.where(
    X['DayOfWeekClaimed'].notna(), (X['DayOfWeek'] != X['DayOfWeekClaimed']).astype(int), np.nan
)

# Approximate ordinal mappings for grouped categories; these are not exact measurements
vehicle_age_map = {'new': 0, '2 years': 2, '3 years': 3, '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7, 'more than 7': 8}
claims_history_map = {'none': 0, '1': 1, '2 to 4': 3, 'more than 4': 5}
supplement_map = {'none': 0, '1 to 2': 1.5, '3 to 5': 4, 'more than 5': 6}
policy_accident_map = {'none': 0, '1 to 7': 4, '8 to 15': 11.5, '15 to 30': 22.5, 'more than 30': 31}
policy_claim_map = {'none': 0, '8 to 15': 11.5, '15 to 30': 22.5, 'more than 30': 31}
address_change_map = {'under 6 months': 0.25, '1 year': 1, '2 to 3 years': 2.5, '4 to 8 years': 6}
# 'no change' is not a duration category, so keep it out of this ordinal mapping.
X['AddressChangedFlag'] = (X['AddressChange_Claim'] != 'no change').astype(int)
vehicle_price_map = {'less than 20000': 1, '20000 to 29000': 2, '30000 to 39000': 3, '40000 to 59000': 4, '60000 to 69000': 5, 'more than 69000': 6}
number_cars_map = {'1 vehicle': 1, '2 vehicles': 2, '3 to 4': 3.5, '5 to 8': 6.5, 'more than 8': 9}

X['VehicleAgeYearsOrdinal'] = X['AgeOfVehicle'].map(vehicle_age_map)
X['PastClaimsOrdinal'] = X['PastNumberOfClaims'].map(claims_history_map)
X['SupplementCountOrdinal'] = X['NumberOfSuppliments'].map(supplement_map)
X['DaysPolicyAccidentOrdinal'] = X['Days_Policy_Accident'].map(policy_accident_map)
X['DaysPolicyClaimOrdinal'] = X['Days_Policy_Claim'].map(policy_claim_map)
X['AddressChangeYearsOrdinal'] = X['AddressChange_Claim'].map(address_change_map)
X['VehiclePriceOrdinal'] = X['VehiclePrice'].map(vehicle_price_map)
X['NumberOfCarsOrdinal'] = X['NumberOfCars'].map(number_cars_map)

X.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AccidentClaimDayMismatch,AddressChangedFlag,VehicleAgeYearsOrdinal,PastClaimsOrdinal,SupplementCountOrdinal,DaysPolicyAccidentOrdinal,DaysPolicyClaimOrdinal,AddressChangeYearsOrdinal,VehiclePriceOrdinal,NumberOfCarsOrdinal
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,1.0,1,3,0,0.0,31.0,31.0,1.0,6,3.5
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,1.0,0,6,0,0.0,31.0,31.0,NaN,6,1.0
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,1.0,0,7,1,0.0,31.0,31.0,NaN,6,1.0
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,1.0,0,8,1,6.0,31.0,31.0,NaN,2,1.0
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,1.0,0,5,0,0.0,31.0,31.0,NaN,6,1.0


## 7. Validate engineered features
Inspect summary statistics, mapping completeness, and the new mismatch features.

In [7]:
engineered_cols = [
    'AgeGroup', 'AccidentClaimMonthMismatch', 'AccidentClaimDayMismatch',
    'VehicleAgeYearsOrdinal', 'PastClaimsOrdinal', 'SupplementCountOrdinal',
    'DaysPolicyAccidentOrdinal', 'DaysPolicyClaimOrdinal',
    'AddressChangeYearsOrdinal', 'VehiclePriceOrdinal', 'NumberOfCarsOrdinal'
]
X[engineered_cols].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AgeGroup,15420,8,41-50,3475,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AccidentClaimMonthMismatch,15419.0,NaN,NaN,NaN,0.257929,0.437509,0.0,0.0,0.0,1.0,1.0
AccidentClaimDayMismatch,15419.0,NaN,NaN,NaN,0.798625,0.401041,0.0,1.0,1.0,1.0,1.0
VehicleAgeYearsOrdinal,15420.0,NaN,NaN,NaN,6.581582,1.501288,0.0,6.0,7.0,8.0,8.0
PastClaimsOrdinal,15420.0,NaN,NaN,NaN,1.950584,1.699188,0.0,0.0,1.0,3.0,5.0
SupplementCountOrdinal,15420.0,NaN,NaN,NaN,2.270006,2.516256,0.0,0.0,1.5,6.0,6.0
DaysPolicyAccidentOrdinal,15420.0,NaN,NaN,NaN,30.768353,2.371102,0.0,31.0,31.0,31.0,31.0
DaysPolicyClaimOrdinal,15420.0,NaN,NaN,NaN,30.940564,0.916013,0.0,31.0,31.0,31.0,31.0
AddressChangeYearsOrdinal,1096.0,NaN,NaN,NaN,4.274179,2.067935,0.25,2.5,6.0,6.0,6.0
VehiclePriceOrdinal,15420.0,NaN,NaN,NaN,2.796109,1.443017,1.0,2.0,2.0,3.0,6.0


In [8]:
mapping_missing = X[engineered_cols].isna().sum().sort_values(ascending=False)
print('Missing values in engineered columns:')
display(mapping_missing[mapping_missing > 0])
print('Mismatch counts:')
display(X[['AccidentClaimMonthMismatch', 'AccidentClaimDayMismatch']].value_counts(dropna=False))

Missing values in engineered columns:


AddressChangeYearsOrdinal     14324
AccidentClaimDayMismatch          1
AccidentClaimMonthMismatch        1
dtype: int64

Mismatch counts:


AccidentClaimMonthMismatch  AccidentClaimDayMismatch
0.0                         1.0                         8875
1.0                         1.0                         3439
0.0                         0.0                         2567
1.0                         0.0                          538
NaN                         NaN                            1
Name: count, dtype: int64

## 8. Final feature review
We retain categorical source fields and numeric/ordinal engineered fields. One-hot encoding, imputation, and scaling will be performed later in model pipelines fitted only on training data.

In [9]:
assert TARGET not in X.columns
num_features = X.select_dtypes(include=['number']).columns.tolist()
cat_features = X.select_dtypes(exclude=['number']).columns.tolist()
print(f'Final feature count: {X.shape[1]}')
print(f'Numeric features: {len(num_features)}')
print(f'Categorical features: {len(cat_features)}')
print('Numeric features:', num_features)
print('Categorical features:', cat_features)

Final feature count: 42
Numeric features: 17
Categorical features: 25
Numeric features: ['WeekOfMonth', 'WeekOfMonthClaimed', 'Age', 'Deductible', 'DriverRating', 'Year', 'AccidentClaimMonthMismatch', 'AccidentClaimDayMismatch', 'AddressChangedFlag', 'VehicleAgeYearsOrdinal', 'PastClaimsOrdinal', 'SupplementCountOrdinal', 'DaysPolicyAccidentOrdinal', 'DaysPolicyClaimOrdinal', 'AddressChangeYearsOrdinal', 'VehiclePriceOrdinal', 'NumberOfCarsOrdinal']
Categorical features: ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'BasePolicy', 'AgeGroup']


## 9. Stratified train/test split
We split before model-specific encoding, imputation, or scaling. Stratification preserves the fraud prevalence approximately in both subsets. The test set will remain untouched during model selection.

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Train features:', X_train.shape, '| Train target:', y_train.shape)
print('Test features:', X_test.shape, '| Test target:', y_test.shape)
print('Train fraud rate:', round(y_train.mean() * 100, 3), '%')
print('Test fraud rate:', round(y_test.mean() * 100, 3), '%')

Train features: (12336, 42) | Train target: (12336,)
Test features: (3084, 42) | Test target: (3084,)
Train fraud rate: 5.982 %
Test fraud rate: 5.999 %


## 10. Save model-development datasets
Save train/test features and targets separately. The baseline-model notebook will load these and build preprocessing/model pipelines. Raw data remains untouched.

In [11]:
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_frame(TARGET).to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_frame(TARGET).to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

engineered_full = X.copy()
engineered_full[TARGET] = y.values
engineered_full.to_csv(PROCESSED_DIR / 'engineered_dataset.csv', index=False)
print('Saved outputs to:', PROCESSED_DIR)
print(sorted(p.name for p in PROCESSED_DIR.glob('*.csv')))

Saved outputs to: c:\Users\rudra\OneDrive\Documents\insurance-claim-fraud-detection\insurance-claim-fraud-detection\data\processed
['X_test.csv', 'X_train.csv', 'engineered_dataset.csv', 'y_test.csv', 'y_train.csv']


## 11. Feature Engineering Conclusions

After successfully running the notebook, the following observations were confirmed:

1\. **Promising engineered features:** Features such as `AgeGroup`, `AccidentClaimMonthMismatch`, `AccidentClaimDayMismatch`, `VehicleAgeYearsOrdinal`, `PastClaimsOrdinal`, `AddressChangedFlag`, `VehiclePriceOrdinal`, and other ordinal-transformed variables were created to capture patterns potentially associated with fraud risk.

2\. **Missing values:** `AddressChangeYearsOrdinal` contains missing values for the *“no change”* category by design, since no duration exists to map. A small number of missing values also exist in the accident–claim mismatch features due to invalid placeholder values in the original data. These will be handled during model preprocessing.

3\. **Features requiring further investigation:** Ordinal mappings for variables such as past claims, vehicle age, supplements, policy-to-accident/claim duration, and address-change duration are approximate transformations. Their usefulness should be evaluated during model training. `PolicyNumber` and `RepNumber` were excluded as identifier-like variables.

4\. **Leakage prevention:** The target variable `FraudFound_P` was separated from the predictors, and identifier-like columns `PolicyNumber` and `RepNumber` were removed. The train/test split was performed before model training, using stratification.

5\. **Train/test consistency:** The fraud prevalence remained consistent across both datasets:

- Training set: **5.982%**
- Test set: **5.999%**

&#x20;  This confirms that stratification successfully preserved the target distribution across the train and test sets.

**Conclusion:** The feature engineering pipeline was successfully executed and validated. The dataset is now ready for baseline model development, with missing values to be handled within the model preprocessing pipeline and engineered features evaluated based on their predictive performance.

### Decisions
- Keep source categorical features for pipeline encoding.
- Treat ordinal mappings as approximate representations of ranges.
- Exclude `PolicyNumber` and `RepNumber` from the initial predictor set.
- Fit encoders, imputers, and scalers only on training data in later pipelines.
- Use stratified 80/20 split, `random_state=42`.